In [10]:
import os
import polars as pl
from tqdm import tqdm
from plotnine import *

In [11]:
os.makedirs("/home/dnanexus/data_dir/bcf_files", exist_ok=True)
os.makedirs("/home/dnanexus/data_dir/parquet_af", exist_ok=True)
os.makedirs("/home/dnanexus/data_dir/parquet_maf", exist_ok=True)

In [17]:
# filename = "norm_qced_ukb24310_c1_b3932_v1"
filename = "norm_qced_ukb24310_c1_b8534_v1"

In [23]:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/bcf_files_qced/batch1_norm/{filename}.bcf -o /home/dnanexus/data_dir/bcf_files/
# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/bcf_files_qced/batch1_norm/{filename}.bcf.bgen -o /home/dnanexus/data_dir/bcf_files/

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_af1e-2.parquet/{filename}.parquet -o /home/dnanexus/data_dir/parquet_af/

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-2.parquet/{filename}.parquet -o /home/dnanexus/data_dir/parquet_maf/

Error: path
"/home/dnanexus/data_dir/bcf_files/norm_qced_ukb24310_c1_b8534_v1.bcf" already
exists but -f/--overwrite was not set
Error: path
"/home/dnanexus/data_dir/parquet_af/norm_qced_ukb24310_c1_b8534_v1.parquet"
already exists but -f/--overwrite was not set
Error: path
"/home/dnanexus/data_dir/parquet_maf/norm_qced_ukb24310_c1_b8534_v1.parquet"
already exists but -f/--overwrite was not set


In [19]:
af_path = f"/home/dnanexus/data_dir/parquet_af/{filename}.parquet"

df_af = pl.read_parquet(af_path).with_columns(
    id = (pl.col("CHROM").cast(pl.Utf8) + ":" + pl.col("POS").cast(pl.Utf8) + ":" + pl.col("REF") + ":" + pl.col("ALT"))
)

df_af['id'].value_counts(sort=True)

id,count
str,u64
"""chr1:170677665:C:T""",6742
"""chr1:170680214:G:A""",6353
"""chr1:170680402:C:T""",6031
"""chr1:170679797:C:T""",5385
"""chr1:170682273:A:C""",5139
…,…
"""chr1:170682308:A:G""",1
"""chr1:170682313:C:T""",1
"""chr1:170682320:T:C""",1


In [20]:
maf_path = f"/home/dnanexus/data_dir/parquet_maf/{filename}.parquet"

df_maf = pl.read_parquet(maf_path).with_columns(
    id = (pl.col("CHROM").cast(pl.Utf8) + ":" + pl.col("POS").cast(pl.Utf8) + ":" + pl.col("REF") + ":" + pl.col("ALT"))
)

df_maf['id'].value_counts(sort=True)

id,count
str,u64
"""chr1:170678173:C:A""",490541
"""chr1:170677665:C:T""",6742
"""chr1:170680214:G:A""",6353
"""chr1:170680402:C:T""",6031
"""chr1:170679797:C:T""",5385
…,…
"""chr1:170682308:A:G""",1
"""chr1:170682313:C:T""",1
"""chr1:170682320:T:C""",1


In [34]:
# !bcftools query -i 'POS=170678173' -f '[%CHROM\t%POS\t%REF\t%ALT\t%SAMPLE\t%GT\n]' /home/dnanexus/data_dir/bcf_files/{filename}.bcf > /home/dnanexus/data_dir/bcf_files/{filename}.tsv

!bcftools query -i 'POS=170677665' -f '[%CHROM\t%POS\t%REF\t%ALT\t%SAMPLE\t%GT\n]' /home/dnanexus/data_dir/bcf_files/{filename}.bcf > /home/dnanexus/data_dir/bcf_files/{filename}.tsv

In [38]:
tsv_path = f"/home/dnanexus/data_dir/bcf_files/{filename}.tsv"
df_tsv = pl.read_csv(tsv_path, has_header=False, separator="\t").rename({
    "column_1": "CHROM",
    "column_2": "POS",
    "column_3": "REF",
    "column_4": "ALT",
    "column_5": "SAMPLE_NAME",
    "column_6": "GT"
}).with_columns(
    id = (pl.col("CHROM").cast(pl.Utf8) + ":" + pl.col("POS").cast(pl.Utf8) + ":" + pl.col("REF") + ":" + pl.col("ALT")),
    # GT = pl.col("column_6").str.split("/").list.get(0).cast(pl.Int8) + pl.col("column_6").str.split("/").list.get(1).cast(pl.Int8)
)
df_tsv

CHROM,POS,REF,ALT,SAMPLE_NAME,GT,id
str,i64,str,str,str,str,str
"""chr1""",170677665,"""C""","""T""","""W000001""","""0/0""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""W000002""","""0/0""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""W000003""","""0/0""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""W000004""","""0/0""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""W000005""","""0/0""","""chr1:170677665:C:T"""
…,…,…,…,…,…,…
"""chr1""",170677665,"""C""","""T""","""3923515""","""0/0""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""1108777""","""0/0""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""3548117""","""0/0""","""chr1:170677665:C:T"""


In [42]:
df_tsv.filter(~pl.col('GT').is_in(['0/0', './.'])).filter(pl.col('id').is_in(df_maf['id'])) #['id'].value_counts(sort=True)

/tmp/ipykernel_18929/2393900970.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


CHROM,POS,REF,ALT,SAMPLE_NAME,GT,id
str,i64,str,str,str,str,str
"""chr1""",170677665,"""C""","""T""","""3306955""","""0/1""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""5437850""","""0/1""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""2166463""","""0/1""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""3587942""","""0/1""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""5498699""","""0/1""","""chr1:170677665:C:T"""
…,…,…,…,…,…,…
"""chr1""",170677665,"""C""","""T""","""5922485""","""0/1""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""3040332""","""0/1""","""chr1:170677665:C:T"""
"""chr1""",170677665,"""C""","""T""","""5137022""","""0/1""","""chr1:170677665:C:T"""


In [43]:
df_maf.filter(pl.col('id').is_in(df_tsv['id']))['id'].value_counts(sort=True)

/tmp/ipykernel_18929/521325446.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


id,count
str,u64
"""chr1:170677665:C:T""",6742


In [44]:
df_af.filter(pl.col('id').is_in(df_tsv['id'].unique()))['id'].value_counts(sort=True)

/tmp/ipykernel_18929/4178919536.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


id,count
str,u64
"""chr1:170677665:C:T""",6742
